# TMol How-To Guide — Advanced Topics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uw-ipd/tmol/blob/master/docs/tutorial/08_advanced_topics.ipynb)

## 8. Advanced Topics

Two unrelated deep-end topics: designing sequences with the packer, and building a `PoseStack` directly from fixed-width atom37 tensors -- the representation most structure-prediction models output -- while keeping the whole thing differentiable.

### Setup

Wheel install, imports, seeding, and device selection live in [notebook_setup.py](https://github.com/uw-ipd/tmol/blob/master/docs/tutorial/notebook_setup.py).

In [1]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlretrieve

    urlretrieve(
        "https://raw.githubusercontent.com/uw-ipd/tmol/"
        "master/docs/tutorial/notebook_setup.py",
        "notebook_setup.py",
    )

from notebook_setup import *  # noqa: F401,F403

device, param_db = setup()


Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository


Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository


### Design with the packer

A `PackerTask` decides, per position, which residue types the packer may place. `restrict_to_repacking` keeps the selected positions at their native identity, and `restrict_absent_name3s` narrows them to a set of residue names; both take a block mask. A *task operation* is just a function that configures a task, so the same design rules can drive a single packing run or every packing step of `fast_relax`. A task needs rotamer samplers too: Dunbrack for residues with chi, `FixedAAChiSampler` for those without (Gly, Ala, AIB), and `IncludeCurrentSampler` to keep the input conformation available.

#### Redesigning the core

In [2]:
cif_text = fetch_cif("1UBQ")
pose_stack = pose_stack_from_file(io.StringIO(cif_text), device, param_db=param_db)
score_function = beta2016_score_function(device, param_db=param_db)

# ubiquitin's buried core, by PDB residue number
core_mask = torch.zeros((1, pose_stack.max_n_blocks), dtype=torch.bool, device=device)
core_residues = torch.tensor([3, 5, 13, 15, 17, 26, 30, 43, 50, 56, 61, 67])
core_mask[0, tmol.blocks_for_residue_labels(pose_stack, 0, "A", core_residues)] = True
samplers = [create_dunbrack_sampler_from_database(param_db, device), FixedAAChiSampler(), IncludeCurrentSampler()]


def design_core(task):
    """Task operation: design the core to hydrophobics and repack everything else."""
    task.restrict_to_repacking(~core_mask)
    task.restrict_absent_name3s({"ALA", "VAL", "LEU", "ILE", "PHE", "MET"}, core_mask)
    for sampler in samplers:
        task.add_conformer_sampler(sampler)


def design_summary(designed, mask):
    """Total score and the mutations within mask, e.g. ILE13VAL."""
    residue_numbers = pose_stack.pdb_info.residue_labels[0]
    mutations = [
        f"{pose_stack.block_type(0, block).name3}{residue_numbers[block]}{designed.block_type(0, block).name3}"
        for block in torch.nonzero(mask[0]).flatten().tolist()
        if designed.block_type(0, block).name3 != pose_stack.block_type(0, block).name3
    ]
    score = float(score_function.render_whole_pose_scoring_module(designed)(designed.coords).sum())
    return f"score {score:.1f}; mutations: {', '.join(mutations) or 'none'}"


task = PackerTask(pose_stack, PackerPalette())
design_core(task)
torch.manual_seed(SEED)
packer_design = pack_rotamers(pose_stack, score_function, task)

core_notes = {"original": design_summary(pose_stack, core_mask), "packer design": design_summary(packer_design, core_mask)}
for label, note in core_notes.items():
    print(f"{label}: {note}")

display(tmol.overlay_view({"original": pose_stack, "packer design": packer_design}, notes=core_notes))

original: score 201.5; mutations: none
packer design: score -90.3; mutations: ILE13VAL, ILE61VAL


/home/runner/work/tmol/tmol/.venv/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:629: DeprecationWarning: The chararray class is deprecated and will be removed in a future release. Use an ndarray with a string or bytes dtype instead.
  record = np.char.array(np.where(array.hetero, "HETATM", "ATOM"))
/home/runner/work/tmol/tmol/.venv/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:629: DeprecationWarning: The chararray class is deprecated and will be removed in a future release. Use an ndarray with a string or bytes dtype instead.
  record = np.char.array(np.where(array.hetero, "HETATM", "ATOM"))
/home/runner/work/tmol/tmol/.venv/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:629: DeprecationWarning: The chararray class is deprecated and will be removed in a future release. Use an ndarray with a string or bytes dtype instead.
  record = np.char.array(np.where(array.hetero, "HETATM", "ATOM"))
/home/runner/work/tmol/tmol/.venv/lib/python3.12/sit

#### Design inside relax

`fast_relax` takes task operations through `task_operations=`; passing `design_core` turns relax into design with backbone flexibility. The relaxed score also includes minimization, so it is not directly comparable to the packer-only score above.

In [3]:
# the same task operation, applied at every packing step of relax
torch.manual_seed(SEED)
relax_design = tmol.fast_relax(
    pose_stack.clone(), score_function, PackerPalette(), tmol.CartesianMoveMap(),
    tmol.FoldForest.reasonable_fold_forest(pose_stack), num_repeats=1, task_operations=[design_core],
)

core_notes["design in relax"] = design_summary(relax_design, core_mask)
print(f"design in relax: {core_notes['design in relax']}")

display(tmol.overlay_view(
    {"original": pose_stack, "packer design": packer_design, "design in relax": relax_design}, notes=core_notes
))

design in relax: score -264.1; mutations: VAL5PHE, ILE13VAL, LEU15ILE, VAL17ALA, VAL26ILE, LEU43ILE


/home/runner/work/tmol/tmol/.venv/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:629: DeprecationWarning: The chararray class is deprecated and will be removed in a future release. Use an ndarray with a string or bytes dtype instead.
  record = np.char.array(np.where(array.hetero, "HETATM", "ATOM"))
/home/runner/work/tmol/tmol/.venv/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:629: DeprecationWarning: The chararray class is deprecated and will be removed in a future release. Use an ndarray with a string or bytes dtype instead.
  record = np.char.array(np.where(array.hetero, "HETATM", "ATOM"))
/home/runner/work/tmol/tmol/.venv/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:629: DeprecationWarning: The chararray class is deprecated and will be removed in a future release. Use an ndarray with a string or bytes dtype instead.
  record = np.char.array(np.where(array.hetero, "HETATM", "ATOM"))
/home/runner/work/tmol/tmol/.venv/lib/python3.12/sit

#### Noncanonical candidates

The `PackerPalette` decides which residue types are candidates at all, before any task restriction. The default palette offers only canonical L residues (plus glycine); `PackerPalette(noncanonical_design=True)` also offers noncanonical types such as AIB (alpha-aminoisobutyric acid, a helix-favoring residue) and the D-amino acids. Offered both at helix positions, the packer can choose AIB but should avoid D-alanine, which does not fit a right-handed helix.

In [4]:
# solvent-facing positions on ubiquitin's helix (residues 23-34)
helix_mask = torch.zeros_like(core_mask)
helix_blocks = tmol.blocks_for_residue_labels(pose_stack, 0, "A", torch.tensor([25, 28, 29, 32, 33]))
helix_mask[0, helix_blocks] = True

# the default palette offers only canonical L residues; noncanonical_design adds the rest
pbt = pose_stack.packed_block_types
offered = {}
for palette in (PackerPalette(), PackerPalette(noncanonical_design=True)):
    palette_task = PackerTask(pose_stack, palette)
    considered = palette_task.per_block_considered_block_types[0, int(helix_blocks[0])].tolist()
    offered[palette.noncanonical_design] = {pbt.active_block_types[i].name3 for i in considered if i >= 0}
print("noncanonical_design=True also offers:", sorted(offered[True] - offered[False]))

# offer L-alanine, D-alanine (DAL), AIB, or the native residue at each helix position
helix_task = PackerTask(pose_stack, PackerPalette(noncanonical_design=True))
helix_task.restrict_to_repacking(~helix_mask)
for block in helix_blocks.tolist():
    block_mask = torch.zeros_like(helix_mask)
    block_mask[0, block] = True
    helix_task.restrict_absent_name3s({"ALA", "DAL", "AIB", pose_stack.block_type(0, block).name3}, block_mask)
for sampler in samplers:
    helix_task.add_conformer_sampler(sampler)

torch.manual_seed(SEED)
helix_design = pack_rotamers(pose_stack, score_function, helix_task)

helix_notes = {"original": design_summary(pose_stack, helix_mask), "helix design": design_summary(helix_design, helix_mask)}
for label, note in helix_notes.items():
    print(f"{label}: {note}")

display(tmol.overlay_view({"original": pose_stack, "helix design": helix_design}, notes=helix_notes))

noncanonical_design=True also offers: ['AIB', 'DAL', 'DAR', 'DAS', 'DCY', 'DGL', 'DGN', 'DHI', 'DIL', 'DLE', 'DLY', 'DPN', 'DPR', 'DSG', 'DSN', 'DTH', 'DTR', 'DTY', 'DVA', 'MED']


original: score 201.5; mutations: none
helix design: score -91.2; mutations: ASN25ALA, ASP32AIB


/home/runner/work/tmol/tmol/.venv/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:629: DeprecationWarning: The chararray class is deprecated and will be removed in a future release. Use an ndarray with a string or bytes dtype instead.
  record = np.char.array(np.where(array.hetero, "HETATM", "ATOM"))
/home/runner/work/tmol/tmol/.venv/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:629: DeprecationWarning: The chararray class is deprecated and will be removed in a future release. Use an ndarray with a string or bytes dtype instead.
  record = np.char.array(np.where(array.hetero, "HETATM", "ATOM"))
/home/runner/work/tmol/tmol/.venv/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:629: DeprecationWarning: The chararray class is deprecated and will be removed in a future release. Use an ndarray with a string or bytes dtype instead.
  record = np.char.array(np.where(array.hetero, "HETATM", "ATOM"))
/home/runner/work/tmol/tmol/.venv/lib/python3.12/sit

### Populate a PoseStack from atom37

"atom37" is the fixed-width, 37-atom-slot-per-residue layout most structure-prediction models (AlphaFold-style) use. `tmol.io.pose_stack_from_atom37` builds a `PoseStack` directly from such a tensor plus a `PoseBuildContext`, entirely on tensor ops with no `AtomArray` involved. Because of that, gradients flow straight back from the built pose's coordinates to the atom37 input -- a model's output coordinates can be scored by TMol and the loss backpropagated through pose construction with nothing detached along the way.

We still need an `AtomArray` once here, to resolve chemistry and pull a real structure's canonical-form tensors (`res_types`, `chain_id`, bond records) for re-populating into atom37 slots -- a real caller already has these tensors from its own model and skips this step entirely.

In [5]:
from tmol.io import (
    atom_array_from_file, build_context_from_atom_array, canonical_form_from_biotite,
    pose_stack_from_atom37, atom37_slot_map_for_ordering,
)

atom37_structure = atom_array_from_file(io.StringIO(cif_text))
atom37_context = build_context_from_atom_array(atom37_structure, device, param_db=param_db)
canonical_ordering = atom37_context.canonical_ordering

canonical_form = canonical_form_from_biotite(
    atom37_structure, device, co=canonical_ordering, missing_density_distance_threshold=0.0,
)


def full_atom_layout(canonical_ordering):
    """Every canonical atom gets its own atom37 slot, in order -- the simplest possible layout."""
    return {
        name3: list(canonical_ordering.restypes_ordered_atom_names[name3])
        for name3 in canonical_ordering.restype_io_equiv_classes
    }


def scatter_to_slots(coords, res_types, slot_map):
    """Spread canonical-form coordinates out into atom37 slot order."""
    n_poses, n_res = res_types.shape
    n_slots = slot_map.shape[1]
    is_real = res_types >= 0
    canonical_atom = slot_map[torch.where(is_real, res_types, torch.zeros_like(res_types)).to(torch.int64)]
    routed = (canonical_atom >= 0) & is_real.unsqueeze(-1)
    slots = torch.full((n_poses, n_res, n_slots, 3), float("nan"), dtype=torch.float32, device=coords.device)
    pose_index = torch.arange(n_poses, device=coords.device).reshape(-1, 1, 1).expand(n_poses, n_res, n_slots)
    res_index = torch.arange(n_res, device=coords.device).reshape(1, -1, 1).expand(n_poses, n_res, n_slots)
    slots[routed] = coords[pose_index[routed], res_index[routed], canonical_atom[routed]]
    return slots


slot_map = atom37_slot_map_for_ordering(canonical_ordering, full_atom_layout(canonical_ordering), device)
atom37_coords = scatter_to_slots(canonical_form.coords, canonical_form.res_types, slot_map)
atom37_coords.requires_grad_(True)
print(f"atom37 tensor shape: {tuple(atom37_coords.shape)} (pose, residue, slot, xyz)")


atom37 tensor shape: (1, 76, 37, 3) (pose, residue, slot, xyz)


`pose_stack_from_atom37` takes the atom37 coordinates, a residue-type tensor, a chain-id tensor, and a `PoseBuildContext` -- the same kind of context object built in notebook 4, here without any ligand.

In [6]:
atom37_pose_stack = pose_stack_from_atom37(
    atom37_coords, canonical_form.res_types, canonical_form.chain_id, atom37_context,
    slot_map=slot_map, disulfides=canonical_form.disulfides, cyclic_bonds=canonical_form.cyclic_bonds,
    covalent_bonds=canonical_form.covalent_bonds, res_not_connected=canonical_form.res_not_connected,
)
print(f"n_poses={atom37_pose_stack.n_poses}, max_n_blocks={atom37_pose_stack.max_n_blocks}")
print(f"pose coordinates require grad: {atom37_pose_stack.coords.requires_grad}")

viewer = tmol.view(atom37_pose_stack)
viewer.show()


n_poses=1, max_n_blocks=76
pose coordinates require grad: True


/home/runner/work/tmol/tmol/.venv/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:629: DeprecationWarning: The chararray class is deprecated and will be removed in a future release. Use an ndarray with a string or bytes dtype instead.
  record = np.char.array(np.where(array.hetero, "HETATM", "ATOM"))


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Scoring that pose and calling `.backward()` reaches all the way back to the original atom37 tensor -- the gradient a model's own training loop would need.

In [7]:
atom37_score_function = beta2016_score_function(device, param_db=atom37_context.parameter_database)
atom37_scorer = atom37_score_function.render_whole_pose_scoring_module(atom37_pose_stack)
atom37_total_score = atom37_scorer(atom37_pose_stack.coords)
print(f"score: {float(atom37_total_score[0].detach()):.2f}")

atom37_total_score.sum().backward()
print(f"gradient reached the atom37 input: {torch.count_nonzero(atom37_coords.grad).item() > 0}")


score: 261.03
gradient reached the atom37 input: True
